[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_2/L4.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 2 | BÀI 4


---

# **PHÂN TÍCH THÀNH PHẦN CHÍNH VÀ MÔ HÌNH HÓA LÃI SUẤT**

|  |  |
|:---|:---|
|**Thời gian đọc** |  30 phút |
|**Kiến thức nền** | PCA, học máy  |
|**Từ khóa** |Đường cong lợi suất,  hiệp phương sai, VaR, Hệ số tải |


---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
%pip install -q pandas-datareader pca
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_2/"

*Ở bài trước, ta đã nghiên cứu phân tích thành phần chính (PCA) và lý thuyết toán học đằng sau nó. Trong bài này, ta áp dụng PCA lên lãi suất trái phiếu kho bạc và dùng các thành phần chính thu được để tính Giá trị chịu rủi ro (VaR).*

##  **1. Giới thiệu**

### **1.1 Đường cong lợi suất**

Đường cong lợi suất là đường biểu diễn lãi suất trái phiếu có cùng chất lượng tín dụng nhưng kỳ hạn khác nhau. Đường cong lợi suất có hai khía cạnh:

- Nó phản ánh kỳ vọng trung bình của nhà đầu tư trên thị trường về diễn biến của các trái phiếu ngắn hạn trong tương lai.
- Phần bù kỳ hạn: khi đường cong lợi suất dốc lên, nó cho thấy niềm tin của nhà đầu tư vào thị trường vì họ lạc quan về việc nhận được lợi suất tốt hơn ở trái phiếu dài hạn. Khi đường cong dốc xuống, nhà đầu tư kỳ vọng thời kỳ khó khăn như suy thoái đang ở phía trước. Đường cong lợi suất cũng có thể phẳng, thể hiện lãi suất ngắn hạn và dài hạn không thay đổi.

Các đặc điểm này của đường cong lợi suất có thể được mô hình hóa bằng các thành phần chính.

### **1.2 Giá trị chịu rủi ro (VaR)**

Trong phần này, ta chỉ đề cập một phần nhỏ của VaR sẽ hữu ích cho chủ đề sau.

VaR là một thước đo hữu ích để đo rủi ro thị trường đối với hiệu suất danh mục. VaR có thể hiểu là khoản lỗ lớn nhất có thể xảy ra với một khoảng tin cậy và một khoảng thời gian cho trước. VaR là một giá trị số phân biệt phần đuôi (khoản lỗ) của một phân phối với phần còn lại của phân phối.

VaR ở cấp độ từng vị thế được tính bằng cách nhân độ biến động (độ lệch chuẩn) của lợi suất vị thế, tổng giá trị danh mục, giá trị tuyệt đối của tỷ trọng và điểm $z$ của khoảng tin cậy đã chọn.

Điểm $z$ được dùng vì ta giả định lợi suất tuân theo phân phối chuẩn. Ví dụ, điểm $z$ của VaR một phía $99\%$ có thể tra trong [bảng phân phối chuẩn](https://stattrek.com/online-calculator/normal) và bằng $2,32635$.

Trong bài này ta tập trung vào VaR cấp độ danh mục, được tính như sau:
$$\text{Portfolio Var} = \text{Portfolio Volatility}\times \text{Portfolio Total Value}\times \text{z-score of confidence interval}$$
trong đó độ biến động của danh mục là độ lệch chuẩn của lợi suất trong danh mục của ta.

## **2. Phân tích thành phần chính**

Phân tích thành phần chính có thể dùng để xử lý rủi ro phát sinh từ dữ liệu có các biến tương quan cao. Với cách tiếp cận này, ta xem xét dữ liệu lịch sử mô tả biến động của các biến thị trường và xác định các nhân tố sẽ giải thích những biến động đó.

Ta xét dữ liệu lợi suất với các kỳ hạn 1 tháng, 3 tháng, 6 tháng, 1 năm, 2 năm, 3 năm, 5 năm, 7 năm, 10 năm, 20 năm và 30 năm.

### **2.1 Nạp các gói thư viện**

Ta bắt đầu bằng việc nạp các gói thư viện hỗ trợ giúp ta hoàn thành các nhiệm vụ.

In [ ]:
# Load libraries
# Global Libraries
# Disable the warnings
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import pandas_datareader.data as web
import scipy as sp

# Plotting
import seaborn as sns
from matplotlib import pyplot
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

from sklearn.decomposition import PCA

### **2.2 Nạp dữ liệu**

Ta tải dữ liệu lợi suất của Mỹ, từ lãi suất kỳ hạn 1 tháng đến 30 năm.

In [ ]:
# downloading the data
start = datetime(2002, 6, 30)
end = datetime(2022, 6, 30)
data = [
    "DGS1MO",
    "DGS3MO",
    "DGS6MO",
    "DGS1",
    "DGS2",
    "DGS3",
    "DGS5",
    "DGS7",
    "DGS10",
    "DGS20",
    "DGS30",
]
data = web.DataReader(data, "fred", start, end).dropna(how="all").ffill()

In [ ]:
data.rename(
    columns={
        "DGS1MO": "1m",
        "DGS3MO": "3m",
        "DGS6MO": "6m",
        "DGS1": "1y",
        "DGS2": "2y",
        "DGS3": "3y",
        "DGS5": "5y",
        "DGS7": "7y",
        "DGS10": "10y",
        "DGS20": "20y",
        "DGS30": "30y",
    },
    inplace=True,
)
df = data.copy()
df.head(10)

### **2.3 Phân tích dữ liệu khám phá**

In [ ]:
df.shape

In [ ]:
df2 = df.copy()

Bây giờ ta trực quan hóa diễn biến của các đường cong lợi suất.

In [ ]:
df.plot(figsize=(15, 8))
pyplot.ylabel("Rate")
pyplot.legend(bbox_to_anchor=(1.01, 0.9), loc=2)
pyplot.suptitle(
    "Fig. 1: Yield Curve Movement", fontweight="bold", horizontalalignment="right"
)
pyplot.show()

## **3. Tạo các thành phần chính**

Trong phần này, ta nghiên cứu cách tạo các thành phần chính từ một tập dữ liệu cho trước.

### **3.1 Dùng ma trận hiệp phương sai**

Dữ liệu hằng ngày từ thị trường tài chính thường có hai tính chất

- Nhiễu
- Tín hiệu

Khi áp dụng PCA cho tập dữ liệu, ta có thể trích xuất tín hiệu của nó và tìm lượng dữ liệu tối thiểu giải thích tỷ lệ phần trăm lớn nhất của toàn bộ tập dữ liệu.

Sau đó ta loại bỏ xu hướng của dữ liệu bằng cách chuẩn hóa thành điểm z.

In [ ]:
df_std = (df - df.mean()) / df.std()
df_std.head()

Ta tính ma trận hiệp phương sai của dữ liệu đã chuẩn hóa ở trên.

In [ ]:
cov_matrix_array = np.array(np.cov(df_std, rowvar=False))
cov_matrix_array

In [ ]:
cov_df = pd.DataFrame(cov_matrix_array, columns=df.columns, index=df.columns)
cov_df

Sau đó, thực hiện phân rã trị riêng trên ma trận hiệp phương sai và tìm tỷ lệ phần trăm của các vector riêng trên tổng phương sai.

In [ ]:
# Perform eigendecomposition

eigenvalues, eigenvectors = np.linalg.eig(cov_matrix_array)

# Put data into a DataFrame and save to excel
df_eigval = pd.DataFrame({"Eigenvalues": eigenvalues})

eigenvalues

In [ ]:
# We calculate explained variance

explained_variance = [round(variance / sum(eigenvalues), 3) for variance in eigenvalues]
explained_variance

In [ ]:
# Save output to Excel
columns = [
    "PC1",
    "PC2",
    "PC3",
    "PC4",
    "PC5",
    "PC6",
    "PC7",
    "PC8",
    "PC9",
    "PC10",
    "PC11",
]
df_eigvec = pd.DataFrame(eigenvectors, columns=columns, index=df.columns)

df_eigvec.to_excel("df_eigvec.xlsx")
eigenvectors[0]

In [ ]:
df_eigvec

Các chỉ số trong bảng trên là kỳ hạn của các lãi suất mà ta xét trong bài tập này, còn các cột trong dataframe là các hệ số tải nhân tố mô tả biến động lãi suất.

Tính biến thiên kết hợp của các thành phần:

In [ ]:
from itertools import accumulate

df_eigval["Explained proportion"] = df_eigval["Eigenvalues"] / np.sum(
    df_eigval["Eigenvalues"]
)
df_eigval["Cumulative Explained Variance"] = list(
    accumulate(df_eigval["Explained proportion"])
)

# Format as percentage
df_eigval.style.format({"Explained proportion": "{:.2%}"})
df_eigval.style.format({"Cumulative Explained Variance": "{:.2%}"})

In [ ]:
fig, ax = pyplot.subplots()

pyplot.plot(df_eigvec["PC1"])
pyplot.suptitle(
    "Fig. 2: The First Factor Loading", fontweight="bold", horizontalalignment="right"
)
pyplot.ylabel("Factor Loadings")
ax.set_xticks(np.arange(11))
ax.set_xticklabels(df.columns, rotation="vertical")
pyplot.xlabel("Term")
pyplot.show()

Hệ số tải nhân tố thứ nhất (PC1) là tổ hợp có trọng số của tất cả các lãi suất và biểu diễn sự dịch chuyển song song của đường cong lợi suất. Ta thấy rằng với một đơn vị hệ số tải, lãi suất kỳ hạn 1 tháng tăng $0,299$ điểm cơ bản, lãi suất kỳ hạn 2 tháng tăng $0,302$ điểm cơ bản, và cứ tiếp tục như vậy.

In [ ]:
fig, ax = pyplot.subplots()

pyplot.plot(df_eigvec["PC2"])
pyplot.suptitle(
    "Fig. 3: The Second Factor Loading", fontweight="bold", horizontalalignment="right"
)
pyplot.ylabel("Factor Loadings")
ax.set_xticks(np.arange(11))
ax.set_xticklabels(df.columns, rotation="vertical")
pyplot.xlabel("Term")
pyplot.show()

Hệ số tải nhân tố thứ hai (PC2) biểu diễn sự thay đổi độ dốc của đường cong lợi suất, khi phần đầu ngắn hạn di chuyển ngược chiều với phần cuối dài hạn. Ta thấy các lãi suất từ kỳ hạn 1 tháng đến 4 năm di chuyển theo một hướng, còn các lãi suất từ 5 năm đến 30 năm di chuyển theo hướng ngược lại.

In [ ]:
fig, ax = pyplot.subplots()

pyplot.plot(df_eigvec["PC3"])
pyplot.suptitle(
    "Fig. 4: The Third Factor Loading", fontweight="bold", horizontalalignment="right"
)
pyplot.ylabel("Factor Loadings")


ax.set_xticklabels(df.columns, rotation="vertical")
pyplot.xlabel("Term")
pyplot.show()

Hệ số tải nhân tố thứ ba (PC3) biểu diễn sự xoắn của đường cong lợi suất trái phiếu chính phủ, xảy ra khi các đoạn ngắn hạn và dài hạn cùng tăng trong khi lợi suất ở giữa giảm.

Bây giờ ta có thể vẽ biểu đồ scree để trực quan hóa đóng góp của các hệ số tải nhân tố vào phương sai của tập dữ liệu.

In [ ]:
PC_values = np.arange(11)
pyplot.plot(
    PC_values, df_eigval["Explained proportion"] * 100, "o-", linewidth=2, color="blue"
)
pyplot.plot(
    PC_values,
    df_eigval["Cumulative Explained Variance"] * 100,
    "o-",
    linewidth=2,
    color="red",
)
pyplot.suptitle("Fig. 5: Scree Plot", fontweight="bold", horizontalalignment="right")
pyplot.title("Scree Plot")
pyplot.xlabel("Principal Component")
pyplot.ylabel("Variance Explained")
pyplot.legend(["Individual Variance", "Cumulative Variance"])
pyplot.show()

Ta thấy hai thành phần đầu tiên giải thích hơn $99\%$ phương sai nên có thể dùng để mô tả biến động của đường cong lợi suất.

Bây giờ ta tính các thành phần chính, là tích vô hướng của dữ liệu đã chuẩn hóa và các vector riêng.

In [ ]:
principal_components = df_std.dot(eigenvectors)
principal_components.columns = df_eigvec.columns
principal_components.head()

Bây giờ ta vẽ ba thành phần đầu tiên, giải thích hơn 99% phương sai trong dữ liệu.

In [ ]:
pyplot.plot(principal_components[["PC1", "PC2", "PC3"]])
pyplot.xlabel("Time (Years)")
pyplot.suptitle(
    "Fig. 6: Principal Component", fontweight="bold", horizontalalignment="right"
)
pyplot.legend(["PC1", "PC2", "PC3"], bbox_to_anchor=(1.01, 0.9), loc=2)
pyplot.show()

Bản thân các thành phần chính không cho ta biết nhiều, nhưng từ biểu đồ trên, ta thấy thành phần chính thứ nhất biến động mạnh hơn hai thành phần còn lại, đúng như kỳ vọng.

Dưới đây là một hàm tóm gọn quy trình PCA mà ta đã thấy ở trên.

In [ ]:
def PCA_CALC(df, num_reconstruct):
    df -= df.mean(axis=0) / df.std()
    R = np.cov(df, rowvar=False)
    eigen_values, eigen_vectors = sp.linalg.eigh(R)
    eigen_vectors = eigen_vectors[:, np.argsort(eigen_values)[::-1]]
    eigen_values = eigen_values[np.argsort(eigen_values)[::-1]]
    eigen_vectors = eigen_vectors[:, :num_reconstruct]

    return np.dot(eigen_vectors.T, df.T).T, eigen_values, eigen_vectors

In [ ]:
scores, evals, evecs = PCA_CALC(df, 11)

In [ ]:
fig, ax = pyplot.subplots()

evecs = pd.DataFrame(evecs)
pyplot.plot(evecs.loc[:, 0:2])
pyplot.suptitle(
    "Fig. 7: The Factor Loadings", fontweight="bold", horizontalalignment="right"
)
pyplot.legend(["PC1", "PC2", "PC3"], loc="lower right")
pyplot.ylabel("Factor Loadings")
ax.set_xticks(np.arange(11))
ax.set_xticklabels(df.columns, rotation="vertical")
pyplot.xlabel("Term")
pyplot.show()

Như đã bàn ở phần trước, PC1 phản ánh một sự dịch chuyển (chủ yếu) song song, PC2 phản ánh sự nghiêng, và PC3 phản ánh sự xoắn hay hình cánh bướm.

Bây giờ ta khôi phục dữ liệu gốc từ các hệ số tải nhân tố.

In [ ]:
reconst = pd.DataFrame(np.dot(scores, evecs.T), index=df.index, columns=df.columns)

reconst.plot(figsize=(15, 8))
pyplot.ylabel("Rate")
pyplot.xlabel("Years")
pyplot.title("Reconstructed Mean-Subtracted Dataset")
pyplot.suptitle(
    "Fig. 8: Reconstructed Mean-SUbtracted Dataset",
    fontweight="bold",
    horizontalalignment="right",
)
pyplot.legend(bbox_to_anchor=(1.01, 0.9), loc=2)
pyplot.show()

Bây giờ ta khôi phục tập dữ liệu gốc từ các thành phần chính và vẽ kết quả.

In [ ]:
for cols in reconst.columns:
    reconst[cols] = reconst[cols] + df2.mean(axis=0)[cols]

reconst.plot(figsize=(15, 8))
pyplot.xlabel("Date")
pyplot.ylabel("Rates")
pyplot.title("Reconstructed Initial Dataset")
pyplot.suptitle(
    "Fig. 9: Reconstructed Initial Dataset",
    fontweight="bold",
    horizontalalignment="right",
)
pyplot.legend(bbox_to_anchor=(1.01, 0.9), loc=2)
pyplot.show()

Lưu ý rằng các bước này có thể triển khai bằng một mô-đun scikit learn như dưới đây.

Ta bắt đầu bằng việc tính tương quan giữa các lãi suất theo kỳ hạn.

In [ ]:
# correlation
correlation = df.corr()
pyplot.figure(figsize=(15, 15))
pyplot.title("Correlation Matrix")
pyplot.suptitle(
    "Fig. 10: Correlation Matrix", fontweight="bold", horizontalalignment="right"
)
sns.heatmap(correlation, vmax=1, square=True, annot=True, cmap="cubehelix");

Sau đó, chuẩn hóa tập dữ liệu như đã giải thích ở trên. Lưu ý rằng khi thang đo các biến khác nhau, ta dùng ma trận tương quan, còn khi thang đo các biến tương tự nhau, ta dùng ma trận hiệp phương sai.

In [ ]:
scaler = StandardScaler().fit(df)
rescaleddf = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)
# summarize transformed data
df.dropna(how="any", inplace=True)
rescaleddf.dropna(how="any", inplace=True)
rescaleddf.head(2)

Xem hình trực quan hóa của tập dữ liệu đã co giãn bên dưới.

In [ ]:
rescaleddf.plot(figsize=(14, 10))
pyplot.ylabel("Rate")
pyplot.legend(bbox_to_anchor=(1.01, 0.9), loc=2)
pyplot.suptitle(
    "Fig. 11: Scaled Yield Curve Plot", fontweight="bold", horizontalalignment="right"
)
pyplot.show()

Ta gọi thuật toán PCA và khớp dữ liệu của ta vào đó, sau đó phép biến đổi dữ liệu diễn ra ở đây.

In [ ]:
model = PCA()
PrincipalComponent = model.fit(rescaleddf)

Bây giờ ta trực quan hóa phương sai được giải thích của các thành phần.

In [ ]:
NumEigenvalues = 5
fig, axes = pyplot.subplots(ncols=2, figsize=(14, 4))
pd.Series(model.explained_variance_ratio_[:NumEigenvalues]).sort_values(
    ascending=False
).plot.bar(title="Explained Variance Ratio by Top Factors", ax=axes[0])
pd.Series(model.explained_variance_ratio_[:NumEigenvalues]).cumsum().plot(
    ylim=(0, 1), ax=axes[1], title="Cumulative Explained Variance"
)

# explained_variance
pd.Series(np.cumsum(model.explained_variance_ratio_)).to_frame(
    "Explained Variance_Top 5"
).head(NumEigenvalues).style.format("{:,.2%}".format)
pyplot.suptitle("Fig. 12: Scree Plots", fontweight="bold", horizontalalignment="left")
pyplot.show()

In [ ]:
pyplot.plot(model.components_[0:3].T)
pyplot.xlabel("Principal Component")
pyplot.suptitle(
    "Fig. 13: Factor Loadings", fontweight="bold", horizontalalignment="right"
)
pyplot.legend(["PC1", "PC2", "PC3"], loc="lower right")
pyplot.show()

Một lần nữa, đừng quên rằng PC1 phản ánh một sự dịch chuyển (chủ yếu) song song, PC2 phản ánh sự nghiêng, và PC3 phản ánh sự xoắn hay hình cánh bướm.

Ta tạo một dataframe chỉ gồm 3 thành phần đầu tiên, là điều ta quan tâm.

In [ ]:
model = PCA().fit(rescaleddf)
columns = ["pca_comp_%i" % i for i in range(11)]
df_pca = pd.DataFrame(
    model.transform(rescaleddf), columns=columns, index=rescaleddf.index
)
df_pca.head()

In [ ]:
# get the component variance
# Proportion of Variance (from PC1 to PC11)
model.explained_variance_ratio_

In [ ]:
# Cumulative proportion of variance (from PC1 to PC6)
np.cumsum(model.explained_variance_ratio_)

In [ ]:
# component loadings or weights (correlation coefficient between original variables and the component)
# component loadings represents the elements of the eigenvector
# the squared loadings within the PCs always sums to 1
loadings = model.components_
num_pc = model.n_features_
pc_list = ["PC" + str(i) for i in list(range(1, num_pc + 1))]
loadings_df = pd.DataFrame.from_dict(dict(zip(pc_list, loadings)))
loadings_df["variable"] = df.columns.values
loadings_df = loadings_df.set_index("variable")
loadings_df

In [ ]:
# positive and negative values in component loadings reflects the positive and negative
# correlation of the variables with the PCs. Except A and B, all other variables have
# positive projection on first PC.

# get correlation matrix plot for loadings

import seaborn as sns

sns.set(rc={"figure.figsize": (11.7, 8.27)})
import matplotlib.pyplot as plt

ax = sns.heatmap(loadings_df, annot=True, cmap="Spectral")
pyplot.suptitle(
    "Fig. 14: Correlation matrix for Loadings",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

In [ ]:
# get eigenvalues (variance explained by each PC)
model.explained_variance_

Đến giờ, ba cách tiếp cận mà ta áp dụng đã cho kết quả tương tự. Vì vậy ta muốn vẽ biplot của hai thành phần chính đầu tiên.

In [ ]:
from pca import pca

In [ ]:
# Or reduce the data towards 2 PCs
model = pca(n_components=2)

# Fit transform
results = model.fit_transform(rescaleddf)

In [ ]:
# Make biplot with the number of features
print("\033[1m" + "Fig. 15: Biplot without the scores" + "\033[0m")
fig, ax = model.biplot(cmap=None, label=False, legend=False, figsize=(10, 6))

Từ biplot, ta thấy các lãi suất ngắn hạn có tương quan cao với nhau, và điều này cũng đúng với các lãi suất dài hạn, vì góc giữa chúng nhỏ hơn. So sánh lãi suất ngắn hạn và dài hạn, ta có thể kết luận chúng tương quan âm vì góc giữa chúng rộng hơn. Các lãi suất trung hạn dường như không tương quan với cả lãi suất ngắn hạn lẫn dài hạn.

## **4. Tính Giá trị chịu rủi ro (VaR) bằng PCA**

Trong phần này, ta minh họa cách dùng PCA để tính VaR. Với mục đích của bài, giả sử ta có một danh mục với mức độ nhạy cảm trước biến động lãi suất như trong bảng dưới đây:

In [ ]:
df_portfolio = pd.DataFrame(
    {
        "2 year rate": [10],
        "3 year rate": [4],
        "5 year rate": [-8],
        "7 year rate": [-7],
        "10 year rate": [2],
    }
)

df_portfolio

Bảng được đọc như sau: tăng $1$ điểm cơ bản của lãi suất kỳ hạn 2 năm sẽ làm giá trị danh mục của ta tăng $10$ triệu USD; tương tự, ta sẽ thấy danh mục tăng $4$ triệu USD với lãi suất kỳ hạn 3 năm, và cứ tiếp tục như vậy.

Từ biểu đồ scree, ta thấy hai hệ số tải nhân tố là đủ để mô hình hóa lãi suất của ta vì chúng giải thích hơn $99 \%$ phương sai dữ liệu. Nhớ lại dataframe hệ số tải của ta.

In [ ]:
df_eigvec

Bây giờ ta có thể tính mức độ nhạy cảm do nhân tố thứ nhất gây ra như sau

In [ ]:
10 * 0.3169 + 4 * 0.322 - 8 * 0.3231 - 7 * 0.3143 + 2 * 0.2987

Mức độ nhạy cảm do nhân tố thứ hai gây ra sẽ là

In [ ]:
10 * 0.1856 + 4 * 0.0994 + 8 * 0.0798 + 7 * 0.2122 - 2 * 0.3285

Nhân tố thứ nhất có mức độ nhạy cảm rất nhỏ so với nhân tố thứ hai.

Thay đổi của giá trị danh mục khi đó là
$$\Delta P = 0.2695 f_1 + 3.7204 f_2$$

Nhớ lại phương sai của các hệ số tải dưới đây:

In [ ]:
df_eigval

Ta có thể tính độ lệch chuẩn (điểm số nhân tố) như dưới đây:

In [ ]:
df_eigval["Factor_store"] = np.sqrt(df_eigval["Eigenvalues"])
df_eigval

Độ lệch chuẩn của $\Delta P$ do đó sẽ là

In [ ]:
np.sqrt(np.square(0.2695) * 9.304 + np.square(3.7204) * 1.553)

Cuối cùng, VaR một ngày $99\%$ được cho bởi
$$\sigma(\Delta P) \times Z_{\alpha} $$

In [ ]:
4.7086526 * 2.32635

## **5. Kết luận**

Trong bài này, ta đã áp dụng khái niệm phân tích thành phần chính lên dữ liệu lợi suất và thấy cách đi đến cùng một kết luận bằng ba cách tiếp cận khác nhau. Ta đã dùng kết quả để tính VaR cho danh mục của mình.

**Tài liệu tham khảo**

1. Carcano, Nicola. "Yield Curve Risk Management: Adjusting Principal Component Analysis for Model Errors." *The Journal of Risk*, vol. 12, no. 1, 2009, pp. 3–16.
2. Oprea, Andrea. "The Use of Principal Component Analysis (PCA) in Building Yield Curve Scenarios and Identifying Relative-Value Trading Opportunities on the Romanian Government Bond Market." *Journal of Risk and Financial Management*, vol. 15, no. 6, 2022. https://www.mdpi.com/1911-8074/15/6/247/htm.
3. Stat Trek. "Normal Distribution Calculator." https://stattrek.com/online-calculator/normal.